# 08 — Classificacao de Novos Textos com Ensemble S2

**Objetivo:** Classificar textos ineditos a partir de um arquivo JSONL usando o ensemble de stacking S2
(BERTimbau + LinearSVC → LogisticRegression).

**Entrada:** Arquivo `.jsonl` com campo `analysis_text`.

**Saida:** DataFrame com predicoes (`mercado`/`outros`), scores do meta-classificador e sub-scores de cada metodo.

In [1]:
from pathlib import Path

import pandas as pd

from economy_classifier.predict import load_ensemble, load_texts_from_jsonl, predict

/home/diacrono/Documentos/repositorios/economy-classifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuracao

In [2]:
ENSEMBLE_DIR = Path("../artifacts/ensemble_s2")
INPUT_JSONL = Path("../data/factcheck_scrape_unified.jsonl")  # ajuste para o seu arquivo
TEXT_COLUMN = "analysis_text"
BATCH_SIZE = 16

## 2. Carregar o ensemble S2

In [3]:
ensemble = load_ensemble(ENSEMBLE_DIR)
print(f"Metodos: {ensemble.feature_order}")
print(f"Device:  {ensemble.device}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3167.03it/s]


Metodos: ['bertimbau', 'linearsvc']
Device:  cuda


## 3. Carregar e inspecionar o JSONL

In [4]:
input_df, texts = load_texts_from_jsonl(INPUT_JSONL, text_column=TEXT_COLUMN)
print(f"Textos carregados: {len(texts)}")
print(f"Colunas originais: {list(input_df.columns)}")
input_df.head()

Textos carregados: 45670
Colunas originais: ['record_id', 'source_record_id', 'dataset_id', 'source_url', 'published_at', 'language', 'title', 'author', 'subtitle', 'claim_text', 'body_text', 'analysis_text', 'text_for_ner', 'text_without_stopwords', 'lemmatized_text', 'original_label', 'standard_label', 'category', 'entities', 'variant', 'metadata']


,record_id,source_record_id,dataset_id,source_url,published_at,language,title,author,subtitle,claim_text,...,analysis_text,text_for_ner,text_without_stopwords,lemmatized_text,original_label,standard_label,category,entities,variant,metadata
0,factcheck_scrape_afp_checamos:c5aac01b1a0a9882...,c5aac01b1a0a98826c063dbd6eece653edb403fcd0ba17...,factcheck_scrape_afp_checamos,https://checamos.afp.com/correcoes,NaT,pt-br,Correções,NaN,NaN,Correções,...,correções,correções,correções,correção,NaN,missing,NaN,[],claim_summary,"{'analysis_text_length': 9, 'entity_count': 0,..."
1,factcheck_scrape_afp_checamos:90129823040861be...,90129823040861be6113311c238a506816e77370964653...,factcheck_scrape_afp_checamos,https://checamos.afp.com/doc.afp.com.99X47T3,2026-03-04 16:52:00+00:00,pt-br,Posts usam notícia sobre sumiço de 16 g de urâ...,NaN,NaN,Brasil forneceu ao Irã ampolas de urânio que d...,...,posts usam notícia sobre sumiço de 16 g de urâ...,posts usam notícia sobre sumiço de 16 g de urâ...,posts usam notícia sumiço 16 g urânio alegar f...,post usar notícia sobre sumiço de 16 g de urân...,Falso,false,NaN,"[{'text': 'brasil', 'label': 'LOC', 'start_cha...",claim_summary,"{'analysis_text_length': 755, 'entity_count': ..."
2,factcheck_scrape_afp_checamos:fe153abdc496fe65...,fe153abdc496fe65d07b2d91ba593cce9eea5437fd7b07...,factcheck_scrape_afp_checamos,https://checamos.afp.com/,2026-03-15 01:00:06+00:00,pt-br,Checamos,NaN,NaN,Checamos,...,checamos o fact-checking da afp é um departame...,checamos o fact-checking da afp é um departame...,checamos fact-checking afp departamento agence...,checar o fact-checking de o afp ser um departa...,NaN,missing,NaN,"[{'text': 'afp', 'label': 'ORG', 'start_char':...",claim_summary,"{'analysis_text_length': 275, 'entity_count': ..."
3,factcheck_scrape_afp_checamos:5c5227d82c7fba17...,5c5227d82c7fba1768bdbbd3b8467d2c69cc7b676816d2...,factcheck_scrape_afp_checamos,https://checamos.afp.com/Conheca-equipe,NaT,pt-br,Conheça a equipe,NaN,NaN,Conheça a equipe,...,conheça a equipe,conheça a equipe,conheça equipe,conhecer o equipe,NaN,missing,NaN,[],claim_summary,"{'analysis_text_length': 16, 'entity_count': 0..."
4,factcheck_scrape_afp_checamos:71e9b2e6708db8ba...,71e9b2e6708db8ba3c52fd966335d595ae9fc8145dd1b1...,factcheck_scrape_afp_checamos,https://checamos.afp.com/Normas-eticas-e-edito...,NaT,pt-br,Normas éticas e editoriais da AFP,NaN,NaN,Normas éticas e editoriais da AFP,...,normas éticas e editoriais da afp,normas éticas e editoriais da afp,normas éticas editoriais afp,norma ético e editorial de o afp,NaN,missing,NaN,"[{'text': 'afp', 'label': 'ORG', 'start_char':...",claim_summary,"{'analysis_text_length': 33, 'entity_count': 1..."


## 4. Classificar

In [5]:
results = predict(ensemble, texts, batch_size=BATCH_SIZE)
output = pd.concat(
    [input_df.reset_index(drop=True), results.reset_index(drop=True)],
    axis=1,
)
print(f"Textos classificados: {len(output)}")
output.head(10)

Textos classificados: 45670


,record_id,source_record_id,dataset_id,source_url,published_at,language,title,author,subtitle,claim_text,...,standard_label,category,entities,variant,metadata,y_pred,y_score,label,score_bertimbau,score_linearsvc
0,factcheck_scrape_afp_checamos:c5aac01b1a0a9882...,c5aac01b1a0a98826c063dbd6eece653edb403fcd0ba17...,factcheck_scrape_afp_checamos,https://checamos.afp.com/correcoes,NaT,pt-br,Correções,NaN,NaN,Correções,...,missing,NaN,[],claim_summary,"{'analysis_text_length': 9, 'entity_count': 0,...",0,0.0057,outros,0.0137,0.2896
1,factcheck_scrape_afp_checamos:90129823040861be...,90129823040861be6113311c238a506816e77370964653...,factcheck_scrape_afp_checamos,https://checamos.afp.com/doc.afp.com.99X47T3,2026-03-04 16:52:00+00:00,pt-br,Posts usam notícia sobre sumiço de 16 g de urâ...,NaN,NaN,Brasil forneceu ao Irã ampolas de urânio que d...,...,false,NaN,"[{'text': 'brasil', 'label': 'LOC', 'start_cha...",claim_summary,"{'analysis_text_length': 755, 'entity_count': ...",0,0.0029,outros,0.0005,0.1625
2,factcheck_scrape_afp_checamos:fe153abdc496fe65...,fe153abdc496fe65d07b2d91ba593cce9eea5437fd7b07...,factcheck_scrape_afp_checamos,https://checamos.afp.com/,2026-03-15 01:00:06+00:00,pt-br,Checamos,NaN,NaN,Checamos,...,missing,NaN,"[{'text': 'afp', 'label': 'ORG', 'start_char':...",claim_summary,"{'analysis_text_length': 275, 'entity_count': ...",0,0.2086,outros,0.9832,0.3443
3,factcheck_scrape_afp_checamos:5c5227d82c7fba17...,5c5227d82c7fba1768bdbbd3b8467d2c69cc7b676816d2...,factcheck_scrape_afp_checamos,https://checamos.afp.com/Conheca-equipe,NaT,pt-br,Conheça a equipe,NaN,NaN,Conheça a equipe,...,missing,NaN,[],claim_summary,"{'analysis_text_length': 16, 'entity_count': 0...",0,0.0031,outros,0.0021,0.1744
4,factcheck_scrape_afp_checamos:71e9b2e6708db8ba...,71e9b2e6708db8ba3c52fd966335d595ae9fc8145dd1b1...,factcheck_scrape_afp_checamos,https://checamos.afp.com/Normas-eticas-e-edito...,NaT,pt-br,Normas éticas e editoriais da AFP,NaN,NaN,Normas éticas e editoriais da AFP,...,missing,NaN,"[{'text': 'afp', 'label': 'ORG', 'start_char':...",claim_summary,"{'analysis_text_length': 33, 'entity_count': 1...",0,0.0014,outros,0.0028,0.0227
5,factcheck_scrape_afp_checamos:09e860948283ef88...,09e860948283ef888c991cb2a9308f9704ef0aa7550bbc...,factcheck_scrape_afp_checamos,https://checamos.afp.com/Contato,NaT,pt-br,Contato,NaN,NaN,Contato,...,missing,NaN,[],claim_summary,"{'analysis_text_length': 7, 'entity_count': 0,...",0,0.0035,outros,0.0071,0.1985
6,factcheck_scrape_afp_checamos:5c1ec42f04b8c921...,5c1ec42f04b8c92120489dda40333b1e5000d62ac0b730...,factcheck_scrape_afp_checamos,https://checamos.afp.com/Configuracoes-de-cookies,NaT,pt-br,Configurações de cookies,NaN,NaN,Configurações de cookies,...,missing,NaN,[],claim_summary,"{'analysis_text_length': 24, 'entity_count': 0...",0,0.0067,outros,0.0158,0.3212
7,factcheck_scrape_afp_checamos:49d25d41d14a1128...,49d25d41d14a1128d28e57ba7ea6b07239746786da2cc7...,factcheck_scrape_afp_checamos,https://checamos.afp.com/Manual-de-estilo-de-v...,NaT,pt-br,Manual de estilo de verificação digital da AFP,NaN,NaN,Manual de estilo de verificação digital da AFP,...,missing,NaN,"[{'text': 'afp', 'label': 'ORG', 'start_char':...",claim_summary,"{'analysis_text_length': 46, 'entity_count': 1...",0,0.0113,outros,0.1444,0.3336
8,factcheck_scrape_afp_checamos:298c5a38ca21b347...,298c5a38ca21b3475e066d8779e2ebc88cf869ca7c232f...,factcheck_scrape_afp_checamos,https://checamos.afp.com/sobre-afp,NaT,pt-br,Sobre a AFP,NaN,NaN,Sobre a AFP,...,missing,NaN,"[{'text': 'afp', 'label': 'ORG', 'start_char':...",claim_summary,"{'analysis_text_length': 11, 'entity_count': 1...",0,0.0087,outros,0.0061,0.3814
9,factcheck_scrape_afp_checamos:6b3557d0f4a53a54...,6b3557d0f4a53a54c3762850fe5daa5e58f3357f8a3dc6...,factcheck_scrape_afp_checamos,https://checamos.afp.com/doc.afp.com.99XV363,2026-03-04 19:21:00+00:00,pt-br,É falso que vídeo mostre porta-aviões norte-am...,NaN,NaN,Porta-aviões USS Abraham Lincoln foi atingido ...,...,false,N

## 5. Resumo das predicoes

In [6]:
print(output["label"].value_counts())
print(f"\nScore medio (mercado): {output.loc[output['label'] == 'mercado', 'y_score'].mean():.4f}")
print(f"Score medio (outros):  {output.loc[output['label'] == 'outros', 'y_score'].mean():.4f}")

label
outros     43705
mercado     1965
Name: count, dtype: int64

Score medio (mercado): 0.7053
Score medio (outros):  0.0194


## 6. Sub-scores por metodo

In [7]:
score_cols = [c for c in output.columns if c.startswith("score_")]
output[[TEXT_COLUMN, "label", "y_score"] + score_cols].head(20)

,analysis_text,label,y_score,score_bertimbau,score_linearsvc
0,correções,outros,0.0057,0.0137,0.2896
1,posts usam notícia sobre sumiço de 16 g de urâ...,outros,0.0029,0.0005,0.1625
2,checamos o fact-checking da afp é um departame...,outros,0.2086,0.9832,0.3443
3,conheça a equipe,outros,0.0031,0.0021,0.1744
4,normas éticas e editoriais da afp,outros,0.0014,0.0028,0.0227
5,contato,outros,0.0035,0.0071,0.1985
6,configurações de cookies,outros,0.0067,0.0158,0.3212
7,manual de estilo de verificação digital da afp,outros,0.0113,0.1444,0.3336
8,sobre a afp,outros,0.0087,0.0061,0.3814
9,é falso que vídeo mostre porta-aviões norte-am...,outros,0.0019,0.0004,0.0764


## 7. Exportar resultados

In [8]:
output_path = Path("../artifacts/predictions_s3_factcheck_scrape_unified.csv")
output.to_csv(output_path, index=False)
print(f"Resultados salvos em {output_path}")

Resultados salvos em ../artifacts/predictions_s3_factcheck_scrape_unified.csv
